# 14. Diseno y dimensionamiento del A/B test — Fase 6

**Alcance:** brazo unico, mujeres **<45 alto riesgo** (top-k% del modelo) -> mamografia
proactiva (intervencion) vs atencion estandar (control: cribado inicia a los 45).

**Endpoint primario:** estadio al diagnostico entre casos confirmados (I-IIa temprano vs III-IV
tardio). La intervencion deberia **aumentar la fraccion temprana** / reducir la tardia.

**Lineas base (CAC 2024, verificadas):** estadios I-IIa = **54%**; estadios III-IV = **28,16%**.

Como la incidencia es rara, el cuello de botella es el **numero de casos acumulados** por brazo.
Se barre el tamano del efecto (no se inventa) y se reporta n de casos requerido, leads requeridos
(via VPP del corte) y anos de acumulacion necesarios.


In [1]:
import json, warnings
import numpy as np, pandas as pd
from scipy import stats
warnings.filterwarnings("ignore")

B = "bases"
ALPHA = 0.05; POWER = 0.80
z_a = stats.norm.ppf(1 - ALPHA/2); z_b = stats.norm.ppf(POWER)

# VPP del modelo dentro de <45 por corte: se CARGA del consolidado canonico del pipeline
# (notebook 13, validacion 2023->2024, horizonte 1 ano) -> consistente con el modelo ganador.
RES = json.load(open(f"{B}/_resultados_sin_avicena.json", encoding="utf-8"))
VPP_LT45 = {float(k): {'leads': v['leads'], 'vpp': v['vpp'], 'recall': v['recall']}
            for k, v in RES['operativos']['vpp_lt45'].items()}
BASE_TEMPRANO = 0.54   # I-IIa, CAC 2024 Fig 2.4
print("VPP <45 por corte (canónico):")
for k, v in sorted(VPP_LT45.items()):
    print(f"  top-{k:.0f}%: leads {v['leads']:,} | VPP {v['vpp']*100:.3f}% | recall {v['recall']:.3f}")
print(f"alpha={ALPHA}, power={POWER} | z_a={z_a:.3f} z_b={z_b:.3f}")
print(f"Linea base temprano (I-IIa): {BASE_TEMPRANO*100:.0f}%")


VPP <45 por corte (canónico):
  top-1%: leads 13,810 | VPP 0.753% | recall 0.452
  top-2%: leads 27,620 | VPP 0.460% | recall 0.552
  top-5%: leads 69,051 | VPP 0.220% | recall 0.661
alpha=0.05, power=0.8 | z_a=1.960 z_b=0.842
Linea base temprano (I-IIa): 54%


## 1. Casos requeridos por brazo segun el efecto (estadio temprano)

Test de dos proporciones (control 54% temprano vs intervencion mas alta). n = casos por brazo.


In [2]:
def n_por_brazo(p1, p2):
    pbar = (p1 + p2) / 2
    num = (z_a*np.sqrt(2*pbar*(1-pbar)) + z_b*np.sqrt(p1*(1-p1)+p2*(1-p2)))**2
    return int(np.ceil(num / (p1-p2)**2))

efectos = [0.62, 0.66, 0.70, 0.74, 0.78]  # % temprano en intervencion
rows = []
for p2 in efectos:
    n = n_por_brazo(BASE_TEMPRANO, p2)
    rows.append({'temprano_control%': BASE_TEMPRANO*100, 'temprano_interv%': p2*100,
                 'delta_pp': (p2-BASE_TEMPRANO)*100, 'casos_por_brazo': n, 'casos_total': 2*n})
casos = pd.DataFrame(rows)
pd.set_option('display.width', 200)
print("=== Casos confirmados requeridos por brazo (power 80%, alpha 0.05) ===")
print(casos.to_string(index=False, float_format='{:.1f}'.format))
print("\nEfecto mas pequeno = mas casos. Un +8pp (54->62%) exige muchos casos; +20pp (54->74%) pocos.")


=== Casos confirmados requeridos por brazo (power 80%, alpha 0.05) ===
 temprano_control%  temprano_interv%  delta_pp  casos_por_brazo  casos_total
              54.0              62.0       8.0              597         1194
              54.0              66.0      12.0              261          522
              54.0              70.0      16.0              144          288
              54.0              74.0      20.0               90          180
              54.0              78.0      24.0               60          120

Efecto mas pequeno = mas casos. Un +8pp (54->62%) exige muchos casos; +20pp (54->74%) pocos.


## 2. Leads y anos de acumulacion requeridos por corte

Casos por brazo / VPP = leads por brazo. El pool anual de leads <45 a cada corte limita cuanto
se acumula por ano (asumiendo 1:1 intervencion:control sobre el pool elegible).


In [3]:
rows = []
for k, info in VPP_LT45.items():
    pool, vpp = info['leads'], info['vpp']
    # leads por brazo = pool/2 por ano (split 1:1). casos por brazo por ano = (pool/2)*vpp
    casos_brazo_ano = (pool/2) * vpp
    for p2 in [0.66, 0.70, 0.74]:
        n = n_por_brazo(BASE_TEMPRANO, p2)
        leads_brazo = int(np.ceil(n / vpp))
        anios = leads_brazo / (pool/2)
        rows.append({'top_k%_<45': k, 'VPP%': vpp*100, 'pool_anual': pool,
                     'efecto_interv%': p2*100, 'casos/brazo_req': n,
                     'leads/brazo_req': leads_brazo, 'casos/brazo_por_ano': round(casos_brazo_ano,1),
                     'anios_acum': round(anios,1)})
diseño = pd.DataFrame(rows)
print("=== Factibilidad por corte y efecto ===")
print(diseño.to_string(index=False, float_format='{:.2f}'.format))
print("\nanios_acum = cuantos anos de reclutamiento al pool anual para juntar los casos.")


=== Factibilidad por corte y efecto ===
 top_k%_<45  VPP%  pool_anual  efecto_interv%  casos/brazo_req  leads/brazo_req  casos/brazo_por_ano  anios_acum
       1.00  0.75       13810           66.00              261            34658                52.00        5.00
       1.00  0.75       13810           70.00              144            19122                52.00        2.80
       1.00  0.75       13810           74.00               90            11951                52.00        1.70
       2.00  0.46       27620           66.00              261            56763                63.50        4.10
       2.00  0.46       27620           70.00              144            31318                63.50        2.30
       2.00  0.46       27620           74.00               90            19574                63.50        1.40
       5.00  0.22       69051           66.00              261           118568                76.00        3.40
       5.00  0.22       69051           70.00           

## 3. Lectura y recomendacion de diseno

- Incidencia rara => el endpoint de **estadio** necesita decenas-cientos de casos por brazo;
  con un solo ano de leads <45 puede no alcanzar salvo efectos grandes.
- Opciones para ganar poder: (a) **acumular varios anos** de reclutamiento; (b) **endpoint
  intermedio** mas frecuente (deteccion temprana por 1000 tamizadas, combina incidencia+estadio);
  (c) ampliar el corte top-k% (mas leads, menor VPP -> mas tamizaciones por caso).
- **Unidad de randomizacion:** individual (paciente) maximiza poder (sin efecto de diseno). Si se
  randomiza por regional (cluster), el n se infla por ICC -> generalmente peor; usar solo si hay
  contaminacion operativa entre brazos.


In [4]:
# Endpoint secundario mas potente: tasa de deteccion de cancer (cualquier estadio) por brazo.
# La intervencion adelanta el dx; a 1 ano puede subir la tasa detectada. Test de 2 proporciones
# sobre incidencia detectada entre leads (mucho mas frecuente que el subgrupo de casos-estadio).
print("Endpoint secundario (deteccion): comparar % de leads con dx confirmado interv vs control.")
print("Mas potente porque usa TODOS los leads como denominador, no solo los casos.")
for k, info in VPP_LT45.items():
    pool, vpp = info['leads'], info['vpp']
    # asume intervencion detecta vpp; control detecta menos a 1 ano (lead time). MDE ilustrativo:
    # con pool/2 por brazo, MDE de proporciones alrededor de vpp:
    n_brazo = pool//2
    se = np.sqrt(2*vpp*(1-vpp)/n_brazo)
    mde = (z_a + z_b)*se
    print(f"  top-{k:.0f}%: {n_brazo:,}/brazo, VPP base {vpp*100:.3f}% -> MDE deteccion ~{mde*100:.3f}pp "
          f"(detectable si interv-control >= {mde*100:.3f}pp)")


Endpoint secundario (deteccion): comparar % de leads con dx confirmado interv vs control.
Mas potente porque usa TODOS los leads como denominador, no solo los casos.
  top-1%: 6,905/brazo, VPP base 0.753% -> MDE deteccion ~0.412pp (detectable si interv-control >= 0.412pp)
  top-2%: 13,810/brazo, VPP base 0.460% -> MDE deteccion ~0.228pp (detectable si interv-control >= 0.228pp)
  top-5%: 34,525/brazo, VPP base 0.220% -> MDE deteccion ~0.100pp (detectable si interv-control >= 0.100pp)


## 4. Guardar dimensionamiento

In [5]:
out = {
    'alcance': '<45 alto riesgo -> mamografia proactiva vs estandar',
    'endpoint_primario': 'estadio al diagnostico (I-IIa temprano)',
    'base_temprano_pct': BASE_TEMPRANO*100,
    'fuente_base': 'CAC 2024 Fig 2.4 (I-IIa 54%)',
    'alpha': ALPHA, 'power': POWER,
    'casos_requeridos': casos.to_dict(orient='records'),
    'factibilidad_por_corte': diseño.to_dict(orient='records'),
    'unidad_randomizacion_recomendada': 'individual (paciente)',
    'notas': 'incidencia rara -> considerar acumular varios anos o endpoint de deteccion por 1000',
}
with open(f"{B}/diseno_abtest.json", "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2, ensure_ascii=False, default=float)
print("Guardado: bases/diseno_abtest.json")


Guardado: bases/diseno_abtest.json
